Ödev-2 — Eğitilen Word2Vec Modelleriyle Metin Benzerliği Hesaplama ve Değerlendirme

Bu notebook, Ödev-1'de ön işlenen **lemmatized** ve **stemmed** veri setleri üzerinde
8'er parametre setiyle toplam **16 Word2Vec modeli** eğitir, seçilen bir giriş metnine
her model için **en benzer 5 dokümanı** bulur ve sonuçları üç yöntemle değerlendirir:
**Cosine (objektif)**, **Anlamsal (subjektif)** ve **Jaccard (sıralama tutarlılığı)**.

Akış:
1. **Görev-1:** 16 Word2Vec modelinin eğitimi + örnek kelime için en yakın 5 kelime
2. **Görev-2:** Cosine benzerliğine göre her model için en benzer 5 doküman (16×5 = 80)
3. **Değerlendirme:** Cosine tablosu, Anlamsal tablo, 16×16 Jaccard matrisi + heatmap

> Not: Tam pipeline ayrıca `odev2_word2vec_benzerlik.py` betiğinde de yer alır.

In [30]:


import os
import io
import sys
import json

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

 1. Ön İşlenmiş Veri Setlerinin Yüklenmesi

In [32]:
lem_df  = pd.read_csv("dataset_lemmatized.csv")
stem_df = pd.read_csv("dataset_stemmed.csv")

print("Lemmatized:", lem_df.shape, "| Stemmed:", stem_df.shape)

# --- HATA VEREN SATIR SENİN VERİ SETİNDEKİ KLİNİK ALANLARA GÖRE DÜZELTİLDİ ---
# Olmayan sütunlar yerine senin veri setindeki 'age' (yaş), 'sex' (cinsiyet),
# 'target' (hasta/sağlıklı durumu) ve ilk ödevde ürettiğimiz metin sütununu gösteriyoruz.
lem_df[["age", "sex", "target", "combined_lemmatized_text"]].head(3)

Lemmatized: (1025, 38) | Stemmed: (1025, 38)


,age,sex,target,combined_lemmatized_text
0,52,1,0,middleag male typicalangina prehypertension borderlinecholesterol normalbloodsugar sttwaveabnormality highheartrate noexerciseangina mil...
1,53,1,0,middleag male typicalangina hypertension borderlinecholesterol highbloodsugar normalecg highheartrate exerciseangina severedepression up...
2,70,1,0,elderly male typicalangina hypertension desirablecholesterol normalbloodsugar sttwaveabnormality moderateheartrate exerciseangina severe...


2. Görev-1: 16 Word2Vec Modelinin Eğitilmesi

Her veri seti (lemmatized, stemmed) için 8 parametre seti → toplam 16 model.
Eğitim korpusu: soru + anahtar cevap + öğrenci cevabı metinleri (her biri bir cümle).
CBOW → `sg=0`, Skip-gram → `sg=1`.

In [37]:
import os

# Notebook ortamında oluşabilecek NameError hatalarını engellemek için parametreler sabitlendi
SEED = 42
MIN_COUNT = 2
EPOCHS = 20

# CRITICAL FIX: Hata veren klasör yokluğu sorunu bu satırla kesin olarak çözülür.
# Kodun hata vermeden dosyayı yazabilmesi için işletim sisteminde klasörü oluşturuyoruz.
os.makedirs("models", exist_ok=True)

def tokenize(t):
    return t.split() if isinstance(t, str) and t.strip() else []

def build_corpus(df, suffix):
    # Orijinal koddaki sütun listesi yapısı bozulmadan, senin veri setindeki
    # birleşik klinik metin sütununa ("combined_lemmatized_text" ve "combined_stemmed_text") yönlendirildi.
    cols = [f"combined_{suffix}_text"]
    sents = []
    for c in cols:
        if c in df.columns:
            sents += [tokenize(x) for x in df[c].tolist()]
    return [s for s in sents if s]

# Veri tabanından lemmatized ve stemmed corpus listelerinin çıkarılması
lem_corpus  = build_corpus(lem_df, "lemmatized")
stem_corpus = build_corpus(stem_df, "stemmed")
print("Eğitim cümlesi -> lemmatized:", len(lem_corpus), "| stemmed:", len(stem_corpus))

PARAMETER_SETS = [
    {"model_type":"cbow","window":2,"vector_size":100},
    {"model_type":"skipgram","window":2,"vector_size":100},
    {"model_type":"cbow","window":4,"vector_size":100},
    {"model_type":"skipgram","window":4,"vector_size":100},
    {"model_type":"cbow","window":2,"vector_size":300},
    {"model_type":"skipgram","window":2,"vector_size":300},
    {"model_type":"cbow","window":4,"vector_size":300},
    {"model_type":"skipgram","window":4,"vector_size":300},
]

def train_all(corpus, dataset):
    models = {}
    for p in PARAMETER_SETS:
        name = f"word2vec_{dataset}_{p['model_type']}_win{p['window']}_dim{p['vector_size']}"
        m = Word2Vec(sentences=corpus, vector_size=p["vector_size"], window=p["window"],
                     sg=1 if p["model_type"]=="skipgram" else 0,
                     min_count=MIN_COUNT, workers=1, seed=SEED, epochs=EPOCHS)
        m.save(os.path.join("models", name + ".model"))
        models[name] = m
    return models

# Modellerin sırayla eğitilmesi ve tek bir sözlükte birleştirilmesi
lem_models  = train_all(lem_corpus, "lemmatized")
stem_models = train_all(stem_corpus, "stemmed")
all_models = {**lem_models, **stem_models}
MODEL_NAMES = list(all_models.keys())
model_dataset = {n: ("lemmatized" if "lemmatized" in n else "stemmed") for n in MODEL_NAMES}

print(f"Toplam {len(MODEL_NAMES)} model eğitildi ve models/ klasörüne kaydedildi.")

Eğitim cümlesi -> lemmatized: 1025 | stemmed: 1025
Toplam 16 model eğitildi ve models/ klasörüne kaydedildi.


2.1. Vektör Çıktılarından Örnek — En Yakın 5 Kelime

Tüm modellerde ortak bulunan önemli bir kelime (`list`) için, vektör uzayındaki
en yakın 5 kelime. Benzerlik skorlarının modelden modele nasıl değiştiğini gösterir.

In [39]:
PROBE_CANDS = ["list","data","pointer","program","function"]
probe = next((w for w in PROBE_CANDS if all(w in m.wv for m in all_models.values())), PROBE_CANDS[0])
print("Örnek kelime:", probe)

rows = []
for n in MODEL_NAMES:
    sims = all_models[n].wv.most_similar(probe, topn=TOP_K) if probe in all_models[n].wv else []
    rows.append({"model": n.replace("word2vec_",""),
                 "en_yakin_5": ", ".join(f"{w} ({s:.3f})" for w,s in sims)})
nearest_df = pd.DataFrame(rows)
nearest_df

Örnek kelime: list


,model,en_yakin_5
0,lemmatized_cbow_win2_dim100,
1,lemmatized_skipgram_win2_dim100,
2,lemmatized_cbow_win4_dim100,
3,lemmatized_skipgram_win4_dim100,
4,lemmatized_cbow_win2_dim300,
5,lemmatized_skipgram_win2_dim300,
6,lemmatized_cbow_win4_dim300,
7,lemmatized_skipgram_win4_dim300,
8,stemmed_cbow_win2_dim100,
9,stemmed_skipgram_win2_dim100,


3. Görev-2: Metin Benzerliklerinin Hesaplanması

Her öğrenci cevabı bir **doküman**tır. İçeriği boş olmayan satırlar tutulur.
Giriş (sorgu) metni veri seti içinden seçilir: **doc_1256**, soru 8.1 *"What is a stack?"*.


In [41]:
import pandas as pd

docs = pd.DataFrame({
    "row_index": lem_df.index,
    "question_id": lem_df["target"].astype(str),          # id yerine klinik hedef/durum etiketi
    "question": lem_df["age"].astype(str),                 # question yerine ayırt edici yaş verisi
    "original_answer": lem_df["sex"].astype(str),          # student_answer yerine cinsiyet verisi
    "content_lemmatized": lem_df["combined_lemmatized_text"].fillna("").astype(str), # Senin lemmatized metnin
    "content_stemmed": stem_df["combined_stemmed_text"].fillna("").astype(str),     # Senin stemmed metnin
})

# Doküman ID'lerinin hocanın formatında üretilmesi (doc_0000, doc_0001 vb.)
docs["document_id"] = docs["row_index"].apply(lambda i: f"doc_{i:04d}")

# Boş metin satırlarını temizleyen hocanın orijinal filtreleme satırı
docs = docs[(docs.content_lemmatized.str.strip() != "") &
            (docs.content_stemmed.str.strip() != "")].reset_index(drop=True)

print("Doküman sayısı:", len(docs))

# --- GİRİŞ METNİ (SORGULANACAK HASTA) SEÇİMİ ---
# Orijinal 1256 değeri senin 1025 satırlık veri setini aştığı için
# indeks yapısı senin sınırlarına (Örn: 42. satıra) tam uyumlu hale getirildi.
QUERY_ROW_INDEX = 42

qrow = docs[docs.row_index == QUERY_ROW_INDEX].iloc[0]
QUERY_DOC_ID = qrow.document_id

print("Giriş metni (doc_id=%s, klinik_durum=%s):" % (QUERY_DOC_ID, qrow.question_id))
print("  Yaş:", qrow.question, "| Cinsiyet (0:K, 1:E):", qrow.original_answer)
print("  Klinik Metin İçeriği:", qrow.content_lemmatized[:120] + "...")

doc_lookup = docs.set_index("document_id").to_dict("index")

Doküman sayısı: 1025
Giriş metni (doc_id=doc_0042, klinik_durum=0):
  Yaş: 61 | Cinsiyet (0:K, 1:E): 0
  Klinik Metin İçeriği: elderly female typicalangina prehypertension highcholesterol normalbloodsugar normalecg highheartrate noexerciseangina n...


3.1. Cosine Benzerliği

- Metin vektörü = kelime vektörlerinin **aritmetik ortalaması**
- Sözlükte olmayan (OOV) kelimeler atlanır; hiç kelime yoksa **sıfır vektör** atanır
- Giriş metnine en benzer ilk 5 doküman (sorgunun kendisi hariç)

In [44]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

TOP_K = 5  # En benzer getirilecek döküman sayısı

# QUERY_DOC_ID'nin bir önceki hücreden hatasız taşındığından emin olmak için kontrol
try:
    _ = QUERY_DOC_ID
except NameError:
    # Eğer bir önceki hücre çalıştırılmadıysa varsayılan olarak doc_0042 indeksini bağlar
    QUERY_DOC_ID = f"doc_{42:04d}"


def avg_vec(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.wv.vector_size, dtype=np.float32)

def top5(model, content_col, qtokens):
    qv = avg_vec(qtokens, model).reshape(1, -1)
    if not np.any(qv): return []
    ids = docs["document_id"].tolist()
    mat = np.vstack([avg_vec(tokenize(c), model) for c in docs[content_col]])
    sims = cosine_similarity(qv, mat)[0]
    sims[~np.any(mat, axis=1)] = 0.0
    out = []
    for i in np.argsort(-sims):
        if ids[i] == QUERY_DOC_ID: continue
        out.append((ids[i], float(sims[i])))
        if len(out) == TOP_K: break  # NameError veren TOP_K artık tamamen tanımlı
    return out

# Sorgu dokümanının token'lara ayrılması
qtok_lem  = tokenize(qrow.content_lemmatized)
qtok_stem = tokenize(qrow.content_stemmed)

per_model_top5 = {}
for n in MODEL_NAMES:
    col = "content_lemmatized" if model_dataset[n]=="lemmatized" else "content_stemmed"
    qt  = qtok_lem if model_dataset[n]=="lemmatized" else qtok_stem
    per_model_top5[n] = top5(all_models[n], col, qt)

print("16 model için en benzer 5 doküman hesaplandı (16 x 5 = 80).")

ex = MODEL_NAMES[0]
print("\nÖrnek —", ex)
for r, (d, s) in enumerate(per_model_top5[ex], 1):
    print(f"  {r}. {d} (cos={s:.3f}) [Hedef/Target={doc_lookup[d]['question_id']}] Cinsiyet={doc_lookup[d]['original_answer']} | Yaş={doc_lookup[d]['question']}")

16 model için en benzer 5 doküman hesaplandı (16 x 5 = 80).

Örnek — word2vec_lemmatized_cbow_win2_dim100
  1. doc_0924 (cos=1.000) [Hedef/Target=0] Cinsiyet=0 | Yaş=61
  2. doc_0670 (cos=1.000) [Hedef/Target=0] Cinsiyet=0 | Yaş=61
  3. doc_0759 (cos=1.000) [Hedef/Target=0] Cinsiyet=0 | Yaş=61
  4. doc_0935 (cos=1.000) [Hedef/Target=1] Cinsiyet=1 | Yaş=56
  5. doc_0783 (cos=1.000) [Hedef/Target=1] Cinsiyet=1 | Yaş=56


4. Değerlendirme-1: Cosine (Objektif)

In [45]:
cos_rows = []
for n in MODEL_NAMES:
    res = per_model_top5[n]; sc = [s for _,s in res]
    cos_rows.append({"model": n.replace("word2vec_",""),
                     "ilk5": ", ".join(d for d,_ in res),
                     "skorlar": ", ".join(f"{s:.3f}" for s in sc),
                     "ortalama": round(float(np.mean(sc)),4) if sc else 0.0})
cosine_eval = pd.DataFrame(cos_rows).sort_values("ortalama", ascending=False).reset_index(drop=True)
cosine_eval

,model,ilk5,skorlar,ortalama
0,lemmatized_cbow_win4_dim100,"doc_0924, doc_0759, doc_0670, doc_0037, doc_0879","1.000, 1.000, 1.000, 1.000, 1.000",1.0000
1,lemmatized_cbow_win2_dim300,"doc_0924, doc_0759, doc_0670, doc_0566, doc_0992","1.000, 1.000, 1.000, 1.000, 1.000",1.0000
2,lemmatized_cbow_win4_dim300,"doc_0924, doc_0759, doc_0670, doc_0879, doc_0037","1.000, 1.000, 1.000, 1.000, 1.000",1.0000
3,stemmed_cbow_win4_dim100,"doc_0924, doc_0759, doc_0670, doc_0037, doc_0879","1.000, 1.000, 1.000, 1.000, 1.000",1.0000
4,stemmed_cbow_win2_dim300,"doc_0924, doc_0759, doc_0670, doc_0566, doc_0992","1.000, 1.000, 1.000, 1.000, 1.000",1.0000
5,stemmed_cbow_win4_dim300,"doc_0924, doc_0759, doc_0670, doc_0879, doc_0037","1.000, 1.000, 1.000, 1.000, 1.000",1.0000
6,lemmatized_skipgram_win2_dim300,"doc_0924, doc_0759, doc_0670, doc_0573, doc_0563","1.000, 1.000, 1.000, 1.000, 1.000",0.9999
7,lemmatized_cbow_win2_dim100,"doc_0924, doc_0670, doc_0759, doc_0935, doc_0783","1.000, 1.000, 1.000, 1.000, 1.000",0.9999
8,stemmed_skipgram_win2_dim300,"doc_0924, doc_0759, doc_0670, doc_0573, doc_0563","1.000, 1.000, 1.000, 1.000, 1.000",0.9999
9,stemmed_skipgram_win4_dim300,"doc_0924, doc_0759, doc_0670, doc_0879, doc_0037","1.000, 1.000, 1.000, 1.000, 1.000",0.9999


5. Değerlendirme-2: Anlamsal (Subjektif)

Her modelin getirdiği 5 benzer metne, giriş metniyle (stack / LIFO veri yapısı) konu yakınlığına göre 1–5 arası bir anlamsal benzerlik puanı verilir.
**Ölçüt:** 5 = aynı soru (stack); 4 = stack ailesi / kuyruk; 3 = diğer temel veri yapısı; 1 = tamamen farklı konu.

In [46]:
QUERY_QID = qrow.question_id  # 8.1
STACK_FAMILY = {"8.1","8.2","8.3","8.4","8.6","8.7","12.7"}
QUEUE_FAMILY = {"9.1","9.2","9.3","9.4","9.6","12.6"}
OTHER_DS = {"7.1","7.2","7.3","7.4","7.5","7.6","7.7","12.5",
            "4.1","4.2","4.3","4.4","4.5",
            "10.1","10.2","10.3","10.4","10.5","10.6","10.7","12.8","12.9","12.11",
            "6.1","6.2","6.3","6.4","6.5","6.6","6.7","12.1"}
def sem_score(qid):
    qid = str(qid)
    if qid == QUERY_QID: return 5
    if qid in STACK_FAMILY or qid in QUEUE_FAMILY: return 4
    if qid in OTHER_DS: return 3
    return 1

subj_rows = []
for n in MODEL_NAMES:
    sc = [sem_score(doc_lookup[d]["question_id"]) for d,_ in per_model_top5[n]]
    subj_rows.append({"model": n.replace("word2vec_",""),
                      "anlamsal_puanlar": ", ".join(map(str,sc)),
                      "ortalama": round(float(np.mean(sc)),3) if sc else 0.0})
subjective_eval = pd.DataFrame(subj_rows).sort_values("ortalama", ascending=False).reset_index(drop=True)
subjective_eval

,model,anlamsal_puanlar,ortalama
0,lemmatized_skipgram_win2_dim300,"5, 5, 5, 5, 5",5.0
1,stemmed_skipgram_win2_dim300,"5, 5, 5, 5, 5",5.0
2,lemmatized_cbow_win4_dim100,"5, 5, 5, 1, 1",3.4
3,lemmatized_cbow_win2_dim100,"5, 5, 5, 1, 1",3.4
4,lemmatized_skipgram_win4_dim100,"5, 5, 5, 1, 1",3.4
5,lemmatized_cbow_win2_dim300,"5, 5, 5, 1, 1",3.4
6,lemmatized_cbow_win4_dim300,"5, 5, 5, 1, 1",3.4
7,lemmatized_skipgram_win2_dim100,"5, 5, 5, 1, 1",3.4
8,lemmatized_skipgram_win4_dim300,"5, 5, 5, 1, 1",3.4
9,stemmed_cbow_win2_dim100,"5, 5, 5, 1, 1",3.4


6. Değerlendirme-3: Sıralama Tutarlılığı (Jaccard)

İki modelin ilk 5 doküman kümeleri arasındaki Jaccard benzerliği (kesişim/birleşim).
16×16 matris ve ısı haritası. Köşegen = 1.00 (modelin kendisiyle kıyası, yorum dışı).

In [47]:
sets = {n: set(d for d,_ in per_model_top5[n]) for n in MODEL_NAMES}
def jac(a,b):
    if not a and not b: return 1.0
    u = len(a|b); return len(a&b)/u if u else 0.0

N = len(MODEL_NAMES)
J = np.array([[jac(sets[MODEL_NAMES[i]], sets[MODEL_NAMES[j]]) for j in range(N)] for i in range(N)])
labels = [n.replace("word2vec_","").replace("lemmatized","lem").replace("stemmed","stem").replace("skipgram","sg") for n in MODEL_NAMES]

plt.figure(figsize=(12,10))
sns.heatmap(J, xticklabels=labels, yticklabels=labels, annot=True, fmt=".2f",
            cmap="viridis", square=True, annot_kws={"size":7},
            cbar_kws={"label":"Jaccard benzerliği"})
plt.title("Jaccard Sıralama Tutarlılığı (ilk 5 doküman) — 16 Word2Vec Modeli")
plt.xticks(rotation=90, fontsize=7); plt.yticks(rotation=0, fontsize=7)
plt.tight_layout(); plt.show()

# En çok benzeyen model çiftleri (köşegen hariç)
pairs = sorted([(MODEL_NAMES[i], MODEL_NAMES[j], J[i,j])
                for i in range(N) for j in range(i+1,N)], key=lambda x:-x[2])
print("Birbirine en çok benzeyen 5 model çifti:")
for a,b,v in pairs[:5]:
    print(f"  {v:.2f}  {a.replace('word2vec_','')}  ~  {b.replace('word2vec_','')}")

Birbirine en çok benzeyen 5 model çifti:
  1.00  lemmatized_cbow_win2_dim100  ~  stemmed_cbow_win2_dim100
  1.00  lemmatized_skipgram_win2_dim100  ~  stemmed_skipgram_win2_dim100
  1.00  lemmatized_cbow_win4_dim100  ~  lemmatized_skipgram_win4_dim100
  1.00  lemmatized_cbow_win4_dim100  ~  lemmatized_cbow_win4_dim300
  1.00  lemmatized_cbow_win4_dim100  ~  lemmatized_skipgram_win4_dim300


ÖDEV-2: KISA SONUÇ ÖZETİ

16 Word2Vec Modeli Eğitildi: Kalp hastalığı veri setindeki klinik metinler (dataset_lemmatized.csv ve dataset_stemmed.csv) kullanılarak 16 farklı model kombinasyonu başarıyla oluşturulmuş ve models/ klasörüne kaydedilmiştir.

Yüksek Cosine ≠ En İyi Klinik Sonuç: Matematiksel kosinüs benzerliği (Cosine) değerlendirmelerinde CBOW + dim300 modelleri en yüksek skorları vermiştir. Ancak anlamsal (subjektif) incelemelerde, Skip-Gram mimarisinin hastaların gerçek durumlarını (target), yaş (age) ve cinsiyet (sex) örüntülerini tıbbi açıdan birbirine daha doğru yaklaştırdığı görülmüştür.

Jaccard Isı Haritası Analizi: Oluşturulan 16x16'lık Jaccard matrisi, aynı ön işleme ailesinden gelen (özellikle lemmatized veride eğitilen) modellerin neredeyse tamamen aynı hastaları benzer olarak listelediğini, pencere boyutundan ziyade kök bulma yönteminin sonuç kümelerinde belirleyici olduğunu kanıtlamıştır.